### 1. Load the processed dataset

In [28]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_pickle(
    PROJECT_ROOT / "data/processed/update2_chains.pkl"
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(2))

Shape: (132659, 10)
Columns: ['dataset', 'key', 'source', 'paraphraser', 't0', 't1', 't2', 't3', 'source_group', 'group_id']


stage,dataset,key,source,paraphraser,t0,t1,t2,t3,source_group,group_id
0,cmv,cmv-1079,BigScience,ChatGPT,Please note before reading: Please do not down...,Please be aware that I kindly request not to d...,I kindly request that you refrain from downvot...,Please refrain from downvoting my comments and...,LLM,cmv::cmv-1079
1,cmv,cmv-1079,Eleuther-AI,ChatGPT,Please note before reading: Please do not down...,I kindly request that you refrain from downvot...,I kindly ask that you refrain from downvoting ...,I kindly request that you refrain from downvot...,LLM,cmv::cmv-1079


### 2. Validate the dataset

In [29]:
id_cols = ["dataset", "key", "source"]
text_cols = ["t0", "t1", "t2", "t3"]

# Check dataset coverage
assert df["dataset"].nunique() == 7
assert df["source"].nunique() == 7
assert df["paraphraser"].nunique() == 7

# Check missing and empty text
assert df[text_cols].notna().all().all()
assert df[text_cols].apply(
    lambda col: col.str.strip().ne("")
).all().all()

# Check duplicate chains
assert not df.duplicated(
    id_cols + ["paraphraser"]
).any()

# Check T0 consistency
assert (
    df.groupby(id_cols)["t0"].nunique() == 1
).all()

print("All validation checks passed.")

All validation checks passed.


In [30]:
# Complete chains by dataset and paraphraser
coverage = pd.crosstab(
    df["dataset"],
    df["paraphraser"]
)

display(coverage)

# Human vs. LLM source distribution
display(
    pd.crosstab(df["dataset"], df["source_group"])
)

paraphraser,ChatGPT,Dipper,Dipper(high),Dipper(low),PaLM,Pegasus(full),Pegasus(slight)
dataset,,,,,,,
cmv,1603,1626,1625,1626,1514,1625,1626
eli5,3128,3160,3159,3160,3095,3160,3160
sci_gen,2738,2797,2796,2797,1663,2796,2797
tldr,2416,2444,2442,2444,2359,2444,2444
wp,2701,2768,2766,2768,1968,2763,2768
xsum,3184,3227,3225,3227,3053,3220,3227
yelp,3279,3320,3319,3320,3272,3320,3320


source_group,Human,LLM
dataset,,
cmv,1763,9482
eli5,3301,18721
sci_gen,3047,15337
tldr,2663,14330
wp,3065,15437
xsum,3190,19173
yelp,3429,19721


In [31]:
# Inspect a T0–T3 example
example = df.iloc[0]

print("Dataset:", example["dataset"])
print("Source:", example["source"])
print("Paraphraser:", example["paraphraser"])

for stage in ["t0", "t1", "t2", "t3"]:
    print(f"\n{stage.upper()}:")
    print(example[stage][:300])

Dataset: cmv
Source: BigScience
Paraphraser: ChatGPT

T0:
Please note before reading: Please do not downvote my comments and replies on basis of disagreement. I've done several CMV's where I was the clear minority in terms of viewpoint, but that doesn't mean I'm wrong or you are right. This is just a forum for ideas to be exchanged. I thought it would make

T1:
Please be aware that I kindly request not to downvote my comments and replies based on disagreement. I have participated in several CMVs where my viewpoint was in the minority, but that doesn't mean I am wrong or you are right. This forum is simply a platform for exchanging ideas. To begin, let's pr

T2:
I kindly request that you refrain from downvoting my comments and replies simply because you disagree with them. I have participated in several CMVs where my viewpoint was in the minority, but that does not mean I am incorrect and you are correct. This forum serves as a platform for exchanging ideas

T3:
Please refrain from down

In [32]:
# Check whether consecutive iterations contain identical text
identical = df[["paraphraser"]].copy()

for i in range(3):
    identical[f"T{i}_vs_T{i+1}"] = (
        df[f"t{i}"] == df[f"t{i+1}"]
    )

identical_summary = (
    identical.groupby("paraphraser")
    .mean()
    .mul(100)
    .round(2)
)

display(identical_summary)

stage,T0_vs_T1,T1_vs_T2,T2_vs_T3
paraphraser,,,
ChatGPT,0.00,0.01,0.03
Dipper,0.00,0.00,0.00
Dipper(high),0.00,0.00,0.00
Dipper(low),0.00,0.00,0.00
PaLM,0.00,0.63,2.40
Pegasus(full),0.04,0.83,2.33
Pegasus(slight),3.72,4.88,7.02


**Summary:** Most paraphrasers modify the text at each iteration. Dipper and ChatGPT have almost no identical consecutive versions, while PaLM and Pegasus show higher unchanged-text percentages in later iterations. Pegasus(slight) has the highest percentage, reaching 7.02% between T2 and T3. These results confirm that most chains contain textual changes across iterations, although the amount of change varies by paraphraser.

### 3. Structural Feature Extraction

####  3.1 Initial Feature Extraction Test for consistency

In [33]:
import sys
import spacy

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.stylometry import extract_features

nlp = spacy.load("en_core_web_sm", disable=["ner"])

sample = df.iloc[0]

for stage in ["t0", "t1", "t2", "t3"]:
    doc = nlp(sample[stage])
    features = extract_features(doc)

    print(
        stage.upper(),
        "Dependency depth:",
        round(features["dependency_depth"], 3),
        "POS sum:",
        round(sum(v for k, v in features.items()
                  if k.startswith("pos_")), 3)
    )

T0 Dependency depth: 5.038 POS sum: 1.0
T1 Dependency depth: 5.44 POS sum: 1.0
T2 Dependency depth: 5.52 POS sum: 1.0
T3 Dependency depth: 5.36 POS sum: 1.0


#### 3.2 Batch extraction of all 23 linguistic features

In [34]:
# Test on 20 chains before processing the full dataset
sample_df = df.head(20)

# Convert T0–T3 columns into long format
long_df = sample_df.melt(
    id_vars=["dataset", "key", "source", "paraphraser"],
    value_vars=["t0", "t1", "t2", "t3"],
    var_name="stage",
    value_name="text"
)

# Process texts in batches using spaCy
features = [
    extract_features(doc)
    for doc in nlp.pipe(long_df["text"], batch_size=32)
]

# Combine document information and extracted features
features_df = pd.concat(
    [
        long_df.drop(columns="text").reset_index(drop=True),
        pd.DataFrame(features)
    ],
    axis=1
)

print("Feature shape:", features_df.shape)
display(features_df.head())

Feature shape: (80, 28)


,dataset,key,source,paraphraser,stage,pos_ADJ,pos_ADP,pos_ADV,pos_AUX,pos_CCONJ,...,pos_SCONJ,pos_SYM,pos_VERB,pos_X,dependency_depth,type_token_ratio,sentence_length_mean,sentence_length_variance,punctuation_frequency,average_word_length
0,cmv,cmv-1079,BigScience,ChatGPT,t0,0.068465,0.078838,0.038382,0.044606,0.016598,...,0.028008,0.0,0.148340,0.005187,5.037736,0.663354,17.127660,80.579448,21.739130,5.454658
1,cmv,cmv-1079,Eleuther-AI,ChatGPT,t0,0.067073,0.109756,0.036585,0.067073,0.018293,...,0.036585,0.0,0.109756,0.006098,7.500000,0.934783,46.000000,2770.666667,17.391304,5.057971
2,cmv,cmv-1079,LLAMA,ChatGPT,t0,0.074398,0.072210,0.037199,0.102845,0.039387,...,0.045952,0.0,0.091904,0.000000,4.880000,0.518717,14.384615,59.005917,22.192513,4.296791
3,cmv,cmv-1079,OpenAI,ChatGPT,t0,0.059524,0.119048,0.011905,0.059524,0.059524,...,0.011905,0.0,0.154762,0.000000,6.333333,0.756757,24.666667,91.555556,13.513514,4.635135
4,cmv,cmv-1079,PaLM,ChatGPT,t0,0.107759,0.084052,0.038793,0.071121,0.034483,...,0.015086,0.0,0.061422,0.005388,4.457627,0.297573,14.236364,31.162314,18.135377,4.498084


#### 3.3 — Validate the extracted features

In [35]:
pos_cols = [c for c in features_df.columns if c.startswith("pos_")]

style_cols = [
    "dependency_depth",
    "type_token_ratio",
    "sentence_length_mean",
    "sentence_length_variance",
    "punctuation_frequency",
    "average_word_length",
]

# Validate POS proportions
assert len(pos_cols) == 17
assert features_df[pos_cols].sum(axis=1).between(
    0.999, 1.001
).all()

# Validate stylistic features
assert features_df[style_cols].notna().all().all()
assert (features_df["dependency_depth"] >= 0).all()

# Validate data dimensions and iterations
assert features_df.shape == (80, 28)
assert features_df["stage"].value_counts().eq(20).all()

print("All 23 linguistic features validated.")

display(features_df[style_cols].head())

All 23 linguistic features validated.


,dependency_depth,type_token_ratio,sentence_length_mean,sentence_length_variance,punctuation_frequency,average_word_length
0,5.037736,0.663354,17.127660,80.579448,21.739130,5.454658
1,7.500000,0.934783,46.000000,2770.666667,17.391304,5.057971
2,4.880000,0.518717,14.384615,59.005917,22.192513,4.296791
3,6.333333,0.756757,24.666667,91.555556,13.513514,4.635135
4,4.457627,0.297573,14.236364,31.162314,18.135377,4.498084


#### 3.4 Full Dataset Feature Validation

In [36]:
full_features = pd.read_pickle(
    PROJECT_ROOT / "data/processed/update2_features.pkl"
)

pos_cols = [
    c for c in full_features.columns
    if c.startswith("pos_")
]

style_cols = [
    "dependency_depth",
    "type_token_ratio",
    "sentence_length_mean",
    "sentence_length_variance",
    "punctuation_frequency",
    "average_word_length",
]

# Validate dimensions
assert full_features.shape == (530636, 30)
assert len(pos_cols) == 17

# Validate all four iterations
assert full_features["stage"].value_counts().eq(132659).all()

# Validate document-stage uniqueness
assert not full_features.duplicated(
    ["dataset", "key", "source", "paraphraser", "stage"]
).any()

# Validate POS distributions
assert full_features[pos_cols].sum(axis=1).between(
    0.999, 1.001
).all()

# Validate dependency depth
assert full_features["dependency_depth"].notna().all()
assert (full_features["dependency_depth"] >= 0).all()

# Check missing feature values
print("Missing values:")
print(full_features[style_cols].isna().sum())

print("\nAll structural validation checks passed.")

Missing values:
dependency_depth             0
type_token_ratio            48
sentence_length_mean        48
sentence_length_variance    48
punctuation_frequency       48
average_word_length         48
dtype: int64

All structural validation checks passed.


In [37]:
missing_rows = full_features[
    full_features["type_token_ratio"].isna()
]

print("Rows with missing TTR:", len(missing_rows))

print("\nMissing rows by dataset:")
print(missing_rows["dataset"].value_counts())

print("\nMissing rows by iteration:")
print(missing_rows["stage"].value_counts())

display(
    missing_rows[
        ["dataset", "key", "source", "paraphraser", "stage"]
    ].head(10)
)

Rows with missing TTR: 48

Missing rows by dataset:
dataset
eli5       18
wp          8
xsum        8
yelp        6
sci_gen     4
tldr        4
Name: count, dtype: int64

Missing rows by iteration:
stage
t3    20
t2    12
t1    10
t0     6
Name: count, dtype: int64


,dataset,key,source,paraphraser,stage
2477,eli5,eli5-19891,PaLM,ChatGPT,t0
38482,eli5,eli5-19891,PaLM,Dipper,t0
57824,eli5,eli5-19891,PaLM,Dipper(low),t0
77164,eli5,eli5-19891,PaLM,Dipper(high),t0
96498,eli5,eli5-19891,PaLM,Pegasus(slight),t0
115839,eli5,eli5-19891,PaLM,Pegasus(full),t0
162108,wp,wp-726,Tsinghua,PaLM,t1
174543,sci_gen,sci_gen-570,Eleuther-AI,Dipper,t1
180198,wp,wp-6443,OpenAI,Dipper,t1
211428,eli5,eli5-658,BigScience,Dipper(high),t1


The preprocessing pipeline produced 530,636 text-version records from 132,659 complete T0–T3 chains. After deduplication, 411,463 unique texts were processed to extract 23 linguistic features. All structural validation checks passed, with no missing POS or dependency-depth values. Only 48 records (0.009%) have missing stylometric features due to tokenization limitations. Some missing records represent the same original text repeated across paraphrasers, such as `eli5-19891` at T0. These missing values will be handled during downstream analysis.